In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import sqlite3
from tabulate import tabulate

# 定数
RANKING_PAGE_URL = "https://onsen.nifty.com/rank/year/"
OUTPUT_FILE = "onsen_comments.csv"
DB_FILE = "onsen.db"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/87.0.4280.66 Safari/537.36"
    )
}
def fetch_page(url):
    """指定したURLからHTMLデータを取得し、BeautifulSoupオブジェクトを返す"""
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        return BeautifulSoup(response.text, "html.parser")
    except requests.exceptions.RequestException as e:
        print(f"[ERROR] Failed to fetch: {url}\n{e}")
        return None

def extract_facility_urls(ranking_url, limit=10):
    """ランキングページから指定件数の温泉施設URLを取得"""
    soup = fetch_page(ranking_url)
    if not soup:
        return []
    facility_li_list = soup.select("li.normal, li.small, li.mini")
    facility_urls = [
        li.select_one("a[href]").get("href", "")
        for li in facility_li_list if li.select_one("a[href]")
    ]
    # 絶対URL形式に変換し重複を削除
    facility_urls = [
        href if href.startswith("http") else f"https://onsen.nifty.com{href}"
        for href in facility_urls
    ]
    return list(dict.fromkeys(facility_urls))[:limit]

def scrape_facility_details(facility_url):
    """温泉施設の詳細と口コミを取得"""
    soup = fetch_page(facility_url)
    if not soup:
        return {
            "name": "N/A", "address": "N/A", "tel": "N/A",
            "comment1": "N/A", "comment2": "N/A"
        }

    # テキスト抽出用の関数
    def extract_text(selector):
        element = soup.select_one(selector)
        return element.text.strip() if element else "N/A"

    # 温泉施設情報
    name = extract_text("th:contains('施設名') + td")
    address = extract_text("th:contains('住所') + td")
    tel = extract_text("th:contains('TEL') + td")

    # 口コミを最大2件取得
    comments_section = soup.select("div#kuchikomi div.subSection")
    comments = [
        section.select_one("div.colSet p").get_text(strip=True)
        for section in comments_section[:2] if section.select_one("div.colSet p")
    ]
    comment1 = comments[0] if len(comments) > 0 else "N/A"
    comment2 = comments[1] if len(comments) > 1 else "N/A"

    return {"name": name, "address": address, "tel": tel, "comment1": comment1, "comment2": comment2}

def save_results(data, output_csv=True, output_db=True):
    """結果をCSVとSQLiteデータベースに保存"""
    if output_csv:
        with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["name", "address", "tel", "comment1", "comment2"])
            writer.writeheader()
            writer.writerows(data)
        print(f"[INFO] CSV saved to '{OUTPUT_FILE}'")

    if output_db:
        conn = sqlite3.connect(DB_FILE)
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS onsen_facilities (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT, address TEXT, tel TEXT,
                comment1 TEXT, comment2 TEXT
            )
        """)
        cursor.executemany("""
            INSERT INTO onsen_facilities (name, address, tel, comment1, comment2)
            VALUES (:name, :address, :tel, :comment1, :comment2)
        """, data)
        conn.commit()
        conn.close()
        print(f"[INFO] Data saved to SQLite database '{DB_FILE}'")

def main():
    """ランキングトップ10温泉施設をスクレイピングし結果を保存"""
    print("[INFO] Fetching top 10 facilities from ranking...")
    onsen_urls = extract_facility_urls(RANKING_PAGE_URL, limit=10)
    all_facilities = []

    for url in onsen_urls:
        details = scrape_facility_details(url)
        all_facilities.append(details)
        time.sleep(1)  # サーバー負荷を軽減するためのウェイト

    # 結果の表示
    print("\n--- スクレイピング結果 ---")
    print(tabulate(all_facilities, headers="keys", tablefmt="grid"))

    # 結果を保存
    save_results(all_facilities)

if __name__ == "__main__":
    main()


